## Convert .txt file to structured excel file

In [7]:
import pandas as pd
import re

file_path = "vicoqa-readable-sample.txt"
records = []

with open(file_path, "r", encoding="utf-8-sig") as f:
    content = f.read()

# Split each url block
blocks = re.split(r'={10,}', content)

for block in blocks:
    block = block.strip()
    if not block:
        continue
        
    id_match = re.search(r'ID:\s*(\d+)', block)
    url_match = re.search(r'name:\s*(http[^\n]+)', block)
    if not id_match or not url_match:
        continue
        
    id_val = id_match.group(1).strip()
    url = url_match.group(1).strip()
    
    q_start = re.search(r'\nQ\s+', block)
    if q_start:
        # Extract paragraph
        paragraph_text = block[url_match.end():q_start.start()].strip()
        paragraph = " ".join(paragraph_text.split())
        
        # Extract QA blocks
        qsa_section = block[q_start.start():].strip()
        
        # Seperate each QSA triplet
        qsa_blocks = re.split(r'\nQ\s+', '\n' + qsa_section)
        
        for qsa in qsa_blocks:
            qsa = qsa.strip()
            if not qsa:
                continue
            
            # Split S và A
            s_split = re.split(r'\nS\s+', qsa)
            question = s_split[0].strip()
            
            sentence = ""
            answer = ""
            
            if len(s_split) > 1:
                sa_part = s_split[1]
                a_split = re.split(r'\nA\s+', sa_part)
                sentence = a_split[0].strip()
                if len(a_split) > 1:
                    answer = a_split[1].strip()
            
            question = " ".join(question.split())
            sentence = " ".join(sentence.split())
            answer = " ".join(answer.split())
            
            records.append({
                "id": id_val,
                "url": url,
                "paragraph": paragraph,
                "question": question,
                "sentence": sentence,
                "answer": answer
            })

df = pd.DataFrame(records)
print(f"Number of questions: {len(df)}")

if not df.empty:
    df.to_excel("vicoqa_dataset.xlsx", index=False)


Number of questions: 305


In [8]:
print(df.columns)

Index(['id', 'url', 'paragraph', 'question', 'sentence', 'answer'], dtype='object')


## Statistic

In [9]:
# number of unique URLs
num_urls = df["url"].nunique()

# total number of questions
total_questions = len(df)

# questions per URL
questions_per_url = df.groupby("url").size()

avg_questions = questions_per_url.mean()
min_questions = questions_per_url.min()
max_questions = questions_per_url.max()

print("Number of URLs:", num_urls)
print("Total questions:", total_questions)
print("Questions per URL:")
print("  Avg:", avg_questions)
print("  Min:", min_questions)
print("  Max:", max_questions)

Number of URLs: 61
Total questions: 305
Questions per URL:
  Avg: 5.0
  Min: 5
  Max: 5


## Create .xlsx test file

In [10]:
unq_id = df["id"].unique()
rand_id1 = pd.Series(unq_id).sample(5, random_state = 11).tolist()
print(rand_id1)
rand_id2 = pd.Series(unq_id).sample(5, random_state = 12).tolist()
print(rand_id2)

['888', '873', '275', '897', '1871']
['81', '1690', '873', '983', '1748']


In [11]:
test_records = []

ques_num = 1

for id1, id2 in zip(rand_id1, rand_id2):
    # URL-based
    url_rows = df[df["id"] == id1].head(ques_num) 
    
    for i, (_, row) in enumerate(url_rows.iterrows(), start=1):
        url = row["url"]
        question = row["question"]
        
        message = f"Dựa vào liên kết sau đây: {url}, trả lời câu hỏi: {question}" if i == 1 else question
        
        test_records.append({
            "conversation_id": id1,
            "turn": i,
            "message": message,
            "expected_answer": row["answer"]
        })

    # Paragraph-based
    para_rows = df[df["id"] == id2].head(ques_num)
    
    for j, (_, row) in enumerate(para_rows.iterrows(), start=1):
        paragraph = row["paragraph"]
        question = row["question"]
        
        message = f"Dựa vào đoạn văn sau đây: {paragraph}, trả lời câu hỏi: {question}" if j == 1 else question
        
        test_records.append({
            "conversation_id": id2,
            "turn": j,
            "message": message,
            "expected_answer": row["answer"]
        })
    
    ques_num += 1

In [12]:
test_df = pd.DataFrame(test_records)
print(f"Number of questions: {len(test_df)}")
if not test_df.empty:
    test_df.to_excel("vicoqa_testbot.xlsx", index=False)

Number of questions: 30
